# 01 - CasADi Symbolic Framework

This notebook introduces the core symbolic objects from CasADi documentation Section 3. The goal is to become comfortable with symbolic matrices before using them in QPs and NLPs.

Learning goals:

- Create and inspect `SX`, `MX`, and `DM` matrices.
- Understand shapes, sparsity, indexing, slicing, and concatenation.
- Distinguish elementwise multiplication `*` from matrix multiplication `@`.
- Compute Jacobians, gradients, Hessians, and directional derivatives.

In [1]:
# Colab setup.
# These tutorials assume a fresh Google Colab runtime.
import subprocess
import sys

# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "-q",
#     "casadi",
#     "numpy",
#     "matplotlib",

# ])


In [2]:
import casadi as ca
import numpy as np

print("CasADi version:", ca.__version__)

CasADi version: 3.7.2


## SX Symbols

`SX` is CasADi's scalar-expression symbolic type. An `SX` matrix is a matrix of scalar symbolic expressions.

In [3]:
x = ca.SX.sym("x")
y = ca.SX.sym("y", 5)
Z = ca.SX.sym("Z", 3, 2)

print("x =", x, "shape", x.shape)
print("y =", y, "shape", y.shape)
print("Z =", Z, "shape", Z.shape)

x = x shape (1, 1)
y = [y_0, y_1, y_2, y_3, y_4] shape (5, 1)
Z = 
[[Z_0, Z_3], 
 [Z_1, Z_4], 
 [Z_2, Z_5]] shape (3, 2)


In [4]:
expr_scalar = ca.sqrt(x**2 + 10)
expr_vector = ca.sin(y) + y**2
expr_matrix = Z**2 + 2 * Z + 1

print("scalar expression:", expr_scalar)
print("vector expression:", expr_vector)
print("matrix expression:\n", expr_matrix)

scalar expression: sqrt((sq(x)+10))
vector expression: [(sin(y_0)+sq(y_0)), (sin(y_1)+sq(y_1)), (sin(y_2)+sq(y_2)), (sin(y_3)+sq(y_3)), (sin(y_4)+sq(y_4))]
matrix expression:
 @1=2, @2=1, 
[[((sq(Z_0)+(@1*Z_0))+@2), ((sq(Z_3)+(@1*Z_3))+@2)], 
 [((sq(Z_1)+(@1*Z_1))+@2), ((sq(Z_4)+(@1*Z_4))+@2)], 
 [((sq(Z_2)+(@1*Z_2))+@2), ((sq(Z_5)+(@1*Z_5))+@2)]]


## DM Numeric Matrices

`DM` stores numeric matrices. You will often evaluate symbolic `Function` objects with `DM` or NumPy data.

In [5]:
A = ca.DM([[1, 2, 0], [0, 3, 4]])
b = ca.DM([1, 2, 3])

print("A =\n", A)
print("b =", b)
print("A @ b =", A @ b)
print("as NumPy array:\n", np.array(A))

A =
 
[[1, 2, 0], 
 [0, 3, 4]]
b = [1, 2, 3]
A @ b = [5, 18]
as NumPy array:
 [[1. 2. 0.]
 [0. 3. 4.]]


## Sparse And Dense Matrices

CasADi stores a sparsity pattern separately from expression values. This matters later for solvers.

In [6]:
dense_zero = ca.SX.zeros(3, 3)
sparse_zero = ca.SX(3, 3)
identity = ca.SX.eye(3)

lower_pattern = ca.Sparsity.lower(3)
L = ca.SX.sym("L", lower_pattern)

print("dense_zero sparsity:", dense_zero.sparsity())
print("sparse_zero sparsity:", sparse_zero.sparsity())
print("identity sparsity:", identity.sparsity())
print("lower-triangular symbolic matrix:\n", L)
print("lower pattern:", L.sparsity())

dense_zero sparsity: 3x3
sparse_zero sparsity: 3x3,0nz
identity sparsity: 3x3,3nz
lower-triangular symbolic matrix:
 
[[L_0, 00, 00], 
 [L_1, L_3, 00], 
 [L_2, L_4, L_5]]
lower pattern: 3x3,6nz


## Indexing, Slicing, And Concatenation

CasADi indexing is close to NumPy indexing, but the result is still a symbolic matrix.

In [7]:
M = ca.SX.sym("M", 3, 3)

first_column = M[:, 0]
upper_left = M[0:2, 0:2]
stacked = ca.vertcat(first_column, ca.SX([10, 20, 30]))
wide = ca.horzcat(M, ca.SX.eye(3))

print("first column:", first_column)
print("upper-left block:\n", upper_left)
print("vertical concatenation shape:", stacked.shape)
print("horizontal concatenation shape:", wide.shape)

first column: [M_0, M_1, M_2]
upper-left block:
 
[[M_0, M_3], 
 [M_1, M_4]]
vertical concatenation shape: (6, 1)
horizontal concatenation shape: (3, 6)


## Elementwise `*` Versus Matrix `@`

In CasADi Python, `*` is elementwise multiplication. Use `@` for matrix multiplication.

In [8]:
A_num = ca.DM([[1, 2], [3, 4]])
B_num = ca.DM([[2, 0], [0, 2]])

print("A * B =\n", A_num * B_num)
print("A @ B =\n", A_num @ B_num)

A * B =
 
[[2, 0], 
 [0, 8]]
A @ B =
 
[[2, 4], 
 [6, 8]]


## SX Versus MX

`SX` expands operations scalar-by-scalar. `MX` keeps larger graph operations, which is useful for composing functions and solvers.

In [9]:
X_sx = ca.SX.sym("X", 2, 2)
y_sx = ca.SX.sym("y")
f_sx = 3 * X_sx * X_sx + y_sx

X_mx = ca.MX.sym("X", 2, 2)
y_mx = ca.MX.sym("y")
f_mx = 3 * X_mx * X_mx + y_mx

print("SX expression:\n", f_sx)
print("MX expression:\n", f_mx)

SX expression:
 @1=3, 
[[(((@1*X_0)*X_0)+y), (((@1*X_2)*X_2)+y)], 
 [(((@1*X_1)*X_1)+y), (((@1*X_3)*X_3)+y)]]
MX expression:
 (((3*X)*X)+y)


Do not mix `SX` and `MX` directly in one expression graph. If you need both, wrap the `SX` expression in a `Function` and call that function from an `MX` graph.

In [10]:
f_sx = 3 * X_sx * X_sx + y_sx

try:
    bad = X_sx + X_mx
except TypeError as err:
    print("Expected error when mixing SX and MX directly:")
    print(err)

sx_fun = ca.Function("sx_fun", [X_sx, y_sx], [f_sx])
composed_mx = sx_fun(ca.DM.eye(2), 1.0) + X_mx
print("MX graph containing a call to an SX-defined Function:\n", composed_mx)

Expected error when mixing SX and MX directly:
unsupported operand type(s) for +: 'SX' and 'MX'
MX graph containing a call to an SX-defined Function:
 (
[[4, 1], 
 [1, 4]]+X)


## Automatic Differentiation

CasADi differentiates symbolic expressions exactly by graph transformation.

In [11]:
q = ca.SX.sym("q", 2)
residual = ca.vertcat(
    q[0] + 2 * q[1] - 1,
    ca.sin(q[0]) - q[1]
)
cost = 0.5 * ca.sumsqr(residual)

J = ca.jacobian(residual, q)
grad = ca.gradient(cost, q)
H, grad_from_hessian = ca.hessian(cost, q)

print(cost)
print(J)
print(grad)
print(grad_from_hessian)
print(H)

(0.5*(sq(((q_0+(2*q_1))-1))+sq((sin(q_0)-q_1))))

[[1, 2], 
 [cos(q_0), -1]]
@1=0.5, @2=(sin(q_0)-q_1), @3=(@1*(@2+@2)), @4=((q_0+(2*q_1))-1), @5=(@4+@4), [((cos(q_0)*@3)+(@1*@5)), (@5-@3)]
@1=0.5, @2=(sin(q_0)-q_1), @3=(@1*(@2+@2)), @4=((q_0+(2*q_1))-1), @5=(@4+@4), [((cos(q_0)*@3)+(@1*@5)), (@5-@3)]
@1=cos(q_0), @2=0.5, @3=cos(q_0), @4=(sin(q_0)-q_1), @5=(2-@1), 
[[(((@1*(@2*(@3+@3)))-((@2*(@4+@4))*sin(q_0)))+1), @5], 
 [@5, 5]]


## Exercise

Build a residual for fitting a line to three data points. The model is

$$
\hat y_i(a, b) = a x_i + b.
$$

Stack the decision variables as

$$
\theta = \begin{bmatrix} a \\ b \end{bmatrix}.
$$

The residual vector is

$$
r(\theta) =
\begin{bmatrix}
a x_1 + b - y_1 \\
a x_2 + b - y_2 \\
a x_3 + b - y_3
\end{bmatrix}.
$$

The least-squares problem is

$$
\begin{aligned}
\min_{\theta \in \mathbb{R}^2} \quad
& \frac{1}{2} \|r(\theta)\|_2^2.
\end{aligned}
$$

Compute

$$
J_r(\theta) = \frac{\partial r}{\partial \theta}, \qquad
\nabla f(\theta) = \frac{\partial f}{\partial \theta}, \qquad
\nabla^2 f(\theta) = \frac{\partial^2 f}{\partial \theta^2}.
$$

In [21]:
theta = ca.SX.sym ("theta", 2) # [a, b]
x =  ca.SX.sym ("x", 3)
y = ca.SX.sym ("y", 3)
r = theta[0]*x + theta[1] - y
f = 0.5 * ca.dot(r, r)

J = ca.jacobian (r, theta)
grad = ca.jacobian (f,theta)
H, grad_from_hessian = ca.hessian(f, theta)